# MMS data analysis

Use pySPEDAS to download MMS magnetic-field and FPI plasma-moment data, inspect the products returned for a chosen interval, and make a quick-look plot. The notebook starts with burst data when it is available and otherwise uses fast survey data.

## Requirements

Install ShockGeo with its optional MMS plotting dependencies before running this notebook (pySPEDAS is included in the base install):

```bash
pip install -e ".[mms]"
```

The first download may take a little longer while pySPEDAS obtains data files. Internet access is required.

In [ ]:
from shocklink.mms import (
    average_plotted_values,
    load_mms_data,
    plot_mms_data,
    summarize_data,
)


## Parameters

Edit this cell to select the spacecraft, time interval, and vector coordinate system. Set `MODE` to `"auto"` to prefer burst data and fall back to fast data, `"brst"` to request burst only, or `"fast"` for survey data only. Set `COORDINATES` to `"gse"` (default) or `"gsm"` to convert vector products to time-dependent GSM coordinates. GSM affects magnetic-field and velocity vectors; scalar density and temperature products are unchanged. Short intervals near known burst periods are most likely to return burst data.

In [ ]:
# Edit these values for the MMS interval you want to analyze.
START = "2018-12-19 19:40:00"
END = "2018-12-19 19:52:00"
PROBE = 1
MODE = "auto"  # auto prefers burst, then falls back to fast
COORDINATES = "gse"  # choose "gse" (default) or "gsm"


## Download MMS data

This loads FGM magnetic-field data and FPI ion/electron density, velocity, and temperature moments. The status line reports the cadence and selected coordinate frame.

In [ ]:
data = load_mms_data(
    START, END, probe=PROBE, mode=MODE, coordinates=COORDINATES
)
if not data.series:
    raise RuntimeError(
        "No MMS data found; try MODE = 'fast', a shorter interval, or another time."
    )
print(
    f"Loaded MMS{PROBE} {data.cadence} data ({data.coordinates.upper()}) "
    f"with {len(data.series)} products."
)


## Inspect loaded products

Review which physical quantities were available for this interval. Availability can vary by cadence, spacecraft, and instrument mode.

In [ ]:
for name, series in data.series.items():
    print(f"{name}: {series}")


## Summary statistics

Compute minimum, mean, and maximum values for each loaded component. The plotted averages below include only displayed variables; total temperatures and their averages are in eV.

In [ ]:
summary = summarize_data(data)
summary

averages = average_plotted_values(data)
averages


## Plot data

Create a multi-panel quick-look figure for magnetic field, density, velocity, and temperature. The subtitle includes the interval-averaged GSM spacecraft position in $R_E$ when MEC data are available. Each temperature panel has one line, the left axis in eV, and a linked right axis in K. Missing products are omitted automatically.

In [ ]:
figure = plot_mms_data(data)
figure.show()


## Troubleshooting

- If no data load, try a shorter interval or set `MODE = "fast"`; burst coverage is intermittent.
- Set `COORDINATES = "gsm"` to convert magnetic-field and velocity vectors from GSE to GSM; scalar density and temperature products remain unchanged.
- If imports fail, rerun `pip install -e ".[mms]"` in the same Python environment as Jupyter, then restart the kernel.
- If a product is absent, inspect the loaded-product list; FPI moments are not guaranteed for every requested interval.